# Fallstudie: Supermarktanalyse

**Szenario**: Du bist Data Analyst bei einer deutschen Supermarktkette mit 5 Filialen. Du sollst die Verkaufsdaten des letzten Jahres analysieren, um strategische Empfehlungen für die Geschäftsführung zu erarbeiten.

**Vollständiger DAV-Workflow:**
1. Business Understanding
2. Daten laden & erste Inspektion
3. Datenqualität prüfen
4. Daten transformieren
5. Analyse & Visualisierung
6. Feature Engineering
7. Erkenntnisse & Empfehlungen

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")

In [ ]:
# ── Daten laden von GitHub ────────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
print("Bibliotheken geladen!")

## Phase 1: Business Understanding

**Geschäftsfragen, die wir beantworten wollen:**

1. Welche Produktkategorie generiert den höchsten Umsatz?
2. Welche Filiale performt am besten – und warum?
3. Wie verändert sich das Kaufverhalten im Tagesverlauf?
4. Gibt es Unterschiede zwischen Werktagen und Wochenenden?
5. Welche Produkt-Kategorie-Kombinationen sind je Filiale am umsatzstärksten?

**Datenbasis:** Verkaufstransaktionen aller 5 Filialen, Zeitraum 01.01.2024 – 31.12.2024

In [ ]:
# ── Daten laden ───────────────────────────────────────────────────────────────
df = pd.read_csv(BASE_URL + "supermarkt_verkauf.csv")
print(f"Datensatz geladen: {df.shape[0]:,} Transaktionen, {df.shape[1]} Spalten")
print(f"Zeitraum: {df['datum'].min()} bis {df['datum'].max()}")
print(f"Filialen: {df['filiale'].nunique()} | Kategorien: {df['kategorie'].nunique()}")
print()
df.head(10)

In [ ]:
# ── Erste Inspektion ──────────────────────────────────────────────────────────
print("=== Datentypen ===")
print(df.dtypes)
print()
print("=== Statistische Kennzahlen ===")
df.describe()

## Phase 2: Datenqualität prüfen

Bevor wir analysieren, müssen wir sicherstellen, dass die Daten vollständig und konsistent sind. Wir prüfen:
- Fehlende Werte
- Duplikate
- Plausibilität der Wertebereiche

In [ ]:
# ── Fehlende Werte & Duplikate ────────────────────────────────────────────────
print("=== Fehlende Werte ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Fehlend': missing, 'Anteil (%)': missing_pct})
print(missing_df[missing_df['Fehlend'] > 0] if missing_df['Fehlend'].sum() > 0 else "Keine fehlenden Werte!")
print()
print(f"Duplikate: {df.duplicated().sum()}")
print()
print("=== Werteprüfung ===")
print(f"Negative Preise: {(df['einzelpreis'] < 0).sum()}")
print(f"Negative Mengen: {(df['menge'] < 0).sum()}")
print(f"Zahlungsarten: {df['zahlungsart'].value_counts().to_dict()}")

In [ ]:
# ── Daten transformieren ──────────────────────────────────────────────────────
# Datum als datetime-Objekt
df['datum_obj'] = pd.to_datetime(df['datum'])
df['monat'] = df['datum_obj'].dt.month
df['monat_name'] = df['datum_obj'].dt.strftime('%B')

# Stunde aus Uhrzeit extrahieren
df['stunde'] = df['uhrzeit'].str[:2].astype(int)

# Wochentag-Typ
wochentag_order = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
df['ist_wochenende'] = df['wochentag'].isin(['Samstag', 'Sonntag'])

print("Transformationen abgeschlossen!")
print(f"Neue Spalten: {['datum_obj', 'monat', 'monat_name', 'stunde', 'ist_wochenende']}")
print(f"\nUmsatz gesamt: {df['gesamtpreis'].sum():,.2f} EUR")
print(f"Durchschnittlicher Bon: {df['gesamtpreis'].mean():.2f} EUR")

In [ ]:
# ── Umsatz nach Kategorie ─────────────────────────────────────────────────────
umsatz_kat = (df.groupby('kategorie')['gesamtpreis']
               .sum()
               .reset_index()
               .sort_values('gesamtpreis', ascending=False))
umsatz_kat['anteil_pct'] = (umsatz_kat['gesamtpreis'] / umsatz_kat['gesamtpreis'].sum() * 100).round(1)

fig = px.bar(
    umsatz_kat,
    x='kategorie',
    y='gesamtpreis',
    text='anteil_pct',
    title='Umsatz nach Produktkategorie (2024)',
    labels={'kategorie': 'Kategorie', 'gesamtpreis': 'Umsatz (EUR)'},
    color='gesamtpreis',
    color_continuous_scale='Blues',
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=450)
fig.show()

print(umsatz_kat.to_string(index=False))

In [ ]:
# ── Filial-Performance ────────────────────────────────────────────────────────
filiale_stats = df.groupby('filiale').agg(
    umsatz=('gesamtpreis', 'sum'),
    transaktionen=('gesamtpreis', 'count'),
    durchschn_bon=('gesamtpreis', 'mean')
).reset_index().sort_values('umsatz', ascending=False)
filiale_stats['umsatz'] = filiale_stats['umsatz'].round(2)
filiale_stats['durchschn_bon'] = filiale_stats['durchschn_bon'].round(2)

fig = px.bar(
    filiale_stats,
    x='filiale',
    y='umsatz',
    color='durchschn_bon',
    text='transaktionen',
    title='Filial-Performance: Umsatz und Transaktionsanzahl',
    labels={'filiale': 'Filiale', 'umsatz': 'Gesamtumsatz (EUR)', 'durchschn_bon': 'Ø Bon (EUR)'},
    color_continuous_scale='Teal',
)
fig.update_traces(texttemplate='%{text} Transakt.', textposition='outside')
fig.update_layout(height=450)
fig.show()

print(filiale_stats.to_string(index=False))

In [ ]:
# ── Kaufverhalten nach Tageszeit ──────────────────────────────────────────────
umsatz_stunde = df.groupby('stunde')['gesamtpreis'].agg(['sum', 'count']).reset_index()
umsatz_stunde.columns = ['stunde', 'umsatz', 'transaktionen']

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Umsatz nach Stunde', 'Transaktionen nach Stunde'])

fig.add_trace(
    go.Bar(x=umsatz_stunde['stunde'], y=umsatz_stunde['umsatz'],
           marker_color='steelblue', name='Umsatz'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=umsatz_stunde['stunde'], y=umsatz_stunde['transaktionen'],
           marker_color='coral', name='Transaktionen'),
    row=1, col=2
)
fig.update_layout(title='Kaufverhalten nach Tageszeit', height=400, showlegend=False)
fig.update_xaxes(title_text='Uhrzeit (Stunde)')
fig.show()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
# Jahreszeit
def jahreszeit(monat):
    if monat in [12, 1, 2]:  return 'Winter'
    elif monat in [3, 4, 5]: return 'Frühling'
    elif monat in [6, 7, 8]: return 'Sommer'
    else:                     return 'Herbst'

df['jahreszeit'] = df['monat'].apply(jahreszeit)

# Tageszeit-Kategorie
def tageszeit(stunde):
    if stunde < 10:   return 'Morgen (7-10)'
    elif stunde < 13: return 'Mittag (10-13)'
    elif stunde < 17: return 'Nachmittag (13-17)'
    else:             return 'Abend (17-22)'

df['tageszeit_kat'] = df['stunde'].apply(tageszeit)
df['wochentag_typ'] = df['ist_wochenende'].map({True: 'Wochenende', False: 'Werktag'})

print("Neue Features erstellt:")
print(f"  jahreszeit:  {df['jahreszeit'].value_counts().to_dict()}")
print(f"  tageszeit:   {df['tageszeit_kat'].value_counts().to_dict()}")
print(f"  wochentag:   {df['wochentag_typ'].value_counts().to_dict()}")

# Umsatz nach Jahreszeit
umsatz_jahreszeit = df.groupby('jahreszeit')['gesamtpreis'].sum().reset_index()
fig = px.pie(umsatz_jahreszeit, values='gesamtpreis', names='jahreszeit',
             title='Umsatzverteilung nach Jahreszeit',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
# ── Korrelationsmatrix ────────────────────────────────────────────────────────
num_cols = ['menge', 'einzelpreis', 'gesamtpreis', 'stunde', 'monat']
corr_matrix = df[num_cols].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text}',
))
fig.update_layout(title='Korrelationsmatrix der numerischen Spalten', height=400)
fig.show()

In [ ]:
# ── Top Kategorie-Filiale-Kombinationen ───────────────────────────────────────
pivot = (df.groupby(['filiale', 'kategorie'])['gesamtpreis']
           .sum()
           .reset_index()
           .pivot(index='filiale', columns='kategorie', values='gesamtpreis')
           .fillna(0)
           .round(0))

print("=== Umsatz-Pivot: Filiale × Kategorie (EUR) ===")
print(pivot.to_string())

fig = px.imshow(
    pivot,
    title='Heatmap: Umsatz nach Filiale und Kategorie',
    color_continuous_scale='Blues',
    text_auto='.0f',
    aspect='auto',
)
fig.update_layout(height=400)
fig.show()

## Zusammenfassung & Erkenntnisse

**Wichtigste Ergebnisse:**

1. **Umsatzstärkste Kategorie**: Die Analyse zeigt klar, welche Produktgruppe den größten Beitrag leistet.
2. **Filial-Performance**: Umsatzunterschiede zwischen den Standorten sind messbar – Ursachen könnten Standort, Sortiment oder Öffnungszeiten sein.
3. **Tageszeit-Peaks**: Bestimmte Stunden generieren deutlich mehr Transaktionen – relevant für Personalplanung.
4. **Saisonalität**: Bestimmte Jahreszeiten zeigen höhere Umsätze, was Lager- und Bestellplanung beeinflusst.
5. **Wochenende vs. Werktag**: Analyse ermöglicht gezielte Werbeaktionen.

**Nächste Schritte:**
- Zeitreihenprognose für Monatsumsätze
- Warenkorbanalyse (Market Basket Analysis)
- Churn-Analyse für Kundenkarten-Daten

---
*DAV Lab – Prof. Dr. Robert Butscher – THWS Business School*